# 01 — Ingest Ticketmaster API → Volume

Fetches events, venues, attractions, and classifications from the Ticketmaster
Discovery API and lands raw JSON files in a Unity Catalog Volume.

- Events are fetched in monthly chunks (3 months forward from today)
- Venues, attractions, and classifications are single-batch fetches
- Files land at `/Volumes/{catalog}/{schema}_bronze/raw_data/{entity}/YYYY/MM/DD/*.json`

In [ ]:
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient
from dotenv import load_dotenv
import os, json, time, requests
from datetime import datetime, timedelta
from typing import Dict, List, Optional

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()
w = WorkspaceClient()

In [ ]:
# Configuration
UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "ticketmaster_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

API_KEY  = os.getenv("TICKETMASTER_API_KEY")
BASE_URL = "https://app.ticketmaster.com/discovery/v2"

ENDPOINTS = {
    "events":          "/events.json",
    "venues":          "/venues.json",
    "attractions":     "/attractions.json",
    "classifications": "/classifications.json",
}

PAGE_SIZE  = 200
MAX_PAGES  = 4

if not API_KEY:
    raise ValueError("TICKETMASTER_API_KEY not set in .env")

print(f"Volume path: {VOLUME_PATH}")
print(f"API key loaded ({len(API_KEY)} chars)")

In [ ]:
# Create the UC Volume for raw data landing
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}
""")
print(f"Volume ready: {UC_CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}")

In [ ]:
class TicketmasterAPIClient:
    """Client for Ticketmaster Discovery API with pagination."""

    def __init__(self, api_key: str, base_url: str):
        self.api_key = api_key
        self.base_url = base_url

    def fetch_paginated(self, endpoint: str, page_size: int = 200,
                        max_pages: Optional[int] = None,
                        params: Optional[Dict] = None) -> List[Dict]:
        all_items, page = [], 0
        items_key = self._items_key(endpoint)

        while True:
            if max_pages and page >= max_pages:
                print(f"  Reached max pages limit ({max_pages})")
                break

            req_params = {"apikey": self.api_key, "size": page_size, "page": page}
            if params:
                req_params.update(params)

            resp = requests.get(f"{self.base_url}{endpoint}",
                                params=req_params, timeout=30)
            resp.raise_for_status()
            data = resp.json()

            items = data.get("_embedded", {}).get(items_key, [])
            if not items:
                break

            all_items.extend(items)
            print(f"  Page {page}: {len(items)} items (total: {len(all_items)})")

            total_pages = data.get("page", {}).get("totalPages", 1)
            if page >= total_pages - 1:
                break
            page += 1
            time.sleep(0.2)  # rate-limit courtesy

        return all_items

    @staticmethod
    def _items_key(endpoint: str) -> str:
        for key in ["events", "venues", "attractions", "classifications"]:
            if key in endpoint:
                return key
        return "items"


client = TicketmasterAPIClient(api_key=API_KEY, base_url=BASE_URL)
print("API client ready")

In [ ]:
def write_json_to_volume(items: List[Dict], entity: str, page_num: int = 0) -> str:
    """Write a list of JSON objects to the UC Volume via the SDK files API."""
    now = datetime.utcnow()
    date_path = now.strftime("%Y/%m/%d")
    ts = now.strftime("%Y%m%d_%H%M%S")
    file_path = f"{VOLUME_PATH}/{entity}/{date_path}/{entity}_{ts}_p{page_num}.json"

    payload = json.dumps(items, indent=2).encode("utf-8")
    w.files.upload(file_path, payload, overwrite=True)
    size_mb = len(payload) / (1024 * 1024)
    return file_path, len(items), size_mb

print("Volume writer ready")

In [ ]:
# Fetch events in monthly chunks (3 months forward)
from dateutil.relativedelta import relativedelta

today = datetime.utcnow()
results = {}

print("=" * 60)
print("EVENTS — fetching in monthly chunks")
print("=" * 60)

total_events = []
for month_offset in range(3):
    chunk_start = today + relativedelta(months=month_offset)
    chunk_end   = today + relativedelta(months=month_offset + 1)
    start_str = chunk_start.strftime("%Y-%m-%dT%H:%M:%SZ")
    end_str   = chunk_end.strftime("%Y-%m-%dT%H:%M:%SZ")

    print(f"\nChunk {month_offset + 1}/3: {start_str[:10]} to {end_str[:10]}")
    items = client.fetch_paginated(
        ENDPOINTS["events"], page_size=PAGE_SIZE, max_pages=MAX_PAGES,
        params={"startDateTime": start_str, "endDateTime": end_str, "sort": "date,asc"}
    )
    if items:
        path, count, size = write_json_to_volume(items, "events", page_num=month_offset)
        print(f"  Wrote {count} events ({size:.2f} MB) -> {path}")
        total_events.extend(items)

results["events"] = len(total_events)
print(f"\nTotal events: {len(total_events)}")

In [ ]:
# Fetch venues, attractions, classifications (single batch each)
for entity in ["venues", "attractions", "classifications"]:
    print(f"\nFetching {entity}...")
    items = client.fetch_paginated(
        ENDPOINTS[entity], page_size=PAGE_SIZE, max_pages=MAX_PAGES
    )
    if items:
        path, count, size = write_json_to_volume(items, entity)
        print(f"  Wrote {count} {entity} ({size:.2f} MB) -> {path}")
        results[entity] = count
    else:
        results[entity] = 0
        print(f"  No data found")

In [ ]:
# Summary
print("=" * 60)
print("INGESTION SUMMARY")
print("=" * 60)
for entity, count in results.items():
    print(f"  {entity:20s}: {count:,} records")
print(f"\nVolume path: {VOLUME_PATH}")
print("Ingestion complete — ready for Bronze layer processing.")